# Phishing Attack Detection — Adapted Original Project

This notebook is an adapted version of the selected GitHub project:

**RimTouny / Phishing-Attack-Detection-using-Machine-Learning**

The original notebook loads a processed CSV from a local Windows path, so it cannot be executed directly on another machine.  
With lecturer approval, this version runs the phishing detection workflow on:

`data/dataset_phishing.csv`

Target column in this dataset:

- `status = legitimate` → `label = 0`
- `status = phishing` → `label = 1`

# 1. Set-up

The original notebook installs many packages such as LazyPredict, CatBoost, XGBoost, LightGBM, TensorFlow and scikit-learn 0.24.  
For a reproducible project submission, this adapted notebook uses a stable and focused set of packages from pandas, numpy, matplotlib and scikit-learn.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    BaggingClassifier,
    StackingClassifier,
    VotingClassifier
)
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

RANDOM_STATE = 42
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)

# 2. Data Loading

The original project used:

`E:\data\data_Features (2).csv`

This path works only on the author's computer.  
In this adapted version, the dataset must be placed at:

`data/dataset_phishing.csv`

In [ ]:
DATA_PATH = Path("data/dataset_phishing.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "dataset_phishing.csv was not found. "
        "Please place the CSV file inside the data/ folder as data/dataset_phishing.csv"
    )

Dataset = pd.read_csv(DATA_PATH)
print("Dataset shape:", Dataset.shape)
display(Dataset.head())

# 3. Data Analysis

We inspect the dataset size, data types, missing values, duplicates, and target distribution.

In [ ]:
print("Dataset information:")
Dataset.info()

In [ ]:
print("Number of unique values per column:")
display(Dataset.nunique().sort_values())

In [ ]:
print("Statistical summary:")
display(Dataset.describe(include="all").T)

# 4. Feature Engineering and Data Cleaning

In [ ]:
num_rows, num_columns = Dataset.shape
print("Number of rows:", num_rows)
print("Number of columns:", num_columns)

## Checking for Null Values

In [ ]:
missing = Dataset.isnull().sum().sort_values(ascending=False)
print("Number of NULL values:")
display(missing[missing > 0])

if missing.sum() == 0:
    print("No missing values were found.")

## Checking for Duplicate Rows

In [ ]:
print("Number of duplicate rows:", Dataset.duplicated().sum())

## Target Preparation

The dataset target is `status`.  
We convert it to numeric `label`, as expected by the original project workflow.

In [ ]:
print("Original target distribution:")
display(Dataset["status"].value_counts())

Dataset["label"] = Dataset["status"].map({
    "legitimate": 0,
    "phishing": 1
})

if Dataset["label"].isna().any():
    raise ValueError("Some status values could not be mapped to numeric labels.")

print("Mapped target distribution:")
display(Dataset["label"].value_counts().sort_index())
display((Dataset["label"].value_counts(normalize=True).sort_index() * 100).round(2))

## Feature Matrix

The original notebook expected a `label` column.  
In this dataset, we remove `url` and `status` from the features and use all remaining numeric engineered features.

In [ ]:
# Keep only numeric engineered features.
columns_to_remove = ["url", "status", "label"]
X = Dataset.drop(columns=columns_to_remove, errors="ignore")
X = X.select_dtypes(include=[np.number])
y = Dataset["label"]

# Remove constant columns because they do not help classification.
constant_columns = [col for col in X.columns if X[col].nunique(dropna=False) <= 1]
X = X.drop(columns=constant_columns)

# Fill numeric missing values if any.
X = X.fillna(X.median(numeric_only=True))

print("Removed constant columns:", constant_columns)
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

## Temporal Analysis

The assignment asks for temporal analysis when temporal columns exist.  
This dataset does not include explicit time/date columns, so temporal drift cannot be evaluated directly. This is a limitation because phishing behavior can change over time.

In [ ]:
possible_time_columns = [
    col for col in Dataset.columns
    if any(token in col.lower() for token in ["date", "time", "timestamp", "created", "updated"])
]
print("Possible temporal columns:", possible_time_columns)

if len(possible_time_columns) == 0:
    print("No explicit temporal columns were found.")

## Balance or Non-Balance?

Class prevalence is important in cybersecurity.  
Accuracy can be misleading if one class is much more common than the other.

In [ ]:
class_counts = Dataset["status"].value_counts()

plt.figure(figsize=(5, 4))
plt.bar(class_counts.index.astype(str), class_counts.values)
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()

display(pd.DataFrame({
    "count": class_counts,
    "percent": (class_counts / class_counts.sum() * 100).round(2)
}))

## Outlier Analysis

Cybersecurity data often contains skewed and heavy-tailed features.  
We use IQR to estimate outlier rates because it is more robust than mean and standard deviation.

In [ ]:
def outlier_rate_iqr(series: pd.Series) -> float:
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        return 0.0
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).mean()

outlier_rates = pd.Series({col: outlier_rate_iqr(X[col]) for col in X.columns})
display((outlier_rates.sort_values(ascending=False) * 100).round(2).head(20))

# EDA

## Distribution Graphs

In [ ]:
important_features = [
    "length_url", "length_hostname", "ip", "nb_dots", "nb_hyphens",
    "nb_qm", "nb_eq", "ratio_digits_url", "phish_hints",
    "nb_hyperlinks", "web_traffic", "google_index", "page_rank"
]
important_features = [col for col in important_features if col in X.columns]

for col in important_features:
    plt.figure(figsize=(6, 4))
    plt.hist(Dataset.loc[Dataset["label"] == 0, col].dropna(), bins=40, alpha=0.6, label="legitimate")
    plt.hist(Dataset.loc[Dataset["label"] == 1, col].dropna(), bins=40, alpha=0.6, label="phishing")
    plt.title(f"Distribution of {col} by Class")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.legend()
    plt.show()

## Correlation Matrix

We use Spearman correlation because many URL features are counts, binary variables, or skewed variables.  
Spearman is rank-based and less sensitive to outliers than Pearson.

In [ ]:
spearman_corr = pd.concat([X, y.rename("label")], axis=1).corr(method="spearman")["label"].drop("label")
top_corr = spearman_corr.reindex(spearman_corr.abs().sort_values(ascending=False).index)

display(top_corr.head(20))

plt.figure(figsize=(8, 6))
top_corr.head(15).sort_values().plot(kind="barh")
plt.title("Top Spearman Correlations with Phishing Label")
plt.xlabel("Spearman correlation")
plt.show()

# 5. Modeling

We split the data into train/test sets using stratification to preserve the class distribution.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train distribution:")
display(y_train.value_counts(normalize=True).sort_index())
print("y_test distribution:")
display(y_test.value_counts(normalize=True).sort_index())

## Classification Models

The original project trained many models and compared them.  
Here we use a focused set of models inspired by the original project while keeping the notebook executable and reproducible.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=12,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        max_depth=18,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=RANDOM_STATE
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=120,
        learning_rate=0.08,
        random_state=RANDOM_STATE
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=150,
        max_depth=18,
        min_samples_leaf=2,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE
    ),
    "Gaussian Naive Bayes": GaussianNB(),
    "K Nearest Neighbors": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=5))
    ])
}

In [ ]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = None

    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    result = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_score) if y_score is not None else np.nan,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    print(classification_report(y_test, y_pred, target_names=["legitimate", "phishing"], digits=4))

    ConfusionMatrixDisplay(cm, display_labels=["legitimate", "phishing"]).plot(values_format="d")
    plt.title(f"Confusion Matrix — {name}")
    plt.show()

    return result, model

results = []
trained_models = {}

for name, model in models.items():
    result, fitted_model = evaluate_model(name, model, X_train, X_test, y_train, y_test)
    results.append(result)
    trained_models[name] = fitted_model

results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
display(results_df)

# 6. Champion Model

We select the champion model based on F1-score.  
F1-score is useful here because it balances precision and recall.

In [ ]:
champion_model_name = results_df.iloc[0]["model"]
champion_metrics = results_df.iloc[0]

print("Champion model:", champion_model_name)
display(champion_metrics.to_frame().T)

# 7. Fusion Classifier

The original project uses ensemble/fusion ideas.  
Here we apply a soft voting classifier using the best models from the previous step.

In [ ]:
top_model_names = results_df.head(3)["model"].tolist()
print("Top models used for voting:", top_model_names)

estimators = []
for model_name in top_model_names:
    estimators.append((model_name.replace(" ", "_").lower(), trained_models[model_name]))

voting_classifier = VotingClassifier(
    estimators=estimators,
    voting="soft"
)

voting_result, voting_model = evaluate_model(
    "Soft Voting Ensemble",
    voting_classifier,
    X_train,
    X_test,
    y_train,
    y_test
)

comparison_df = pd.concat([
    results_df,
    pd.DataFrame([voting_result])
], ignore_index=True).sort_values("f1", ascending=False)

display(comparison_df)

# 8. Error Analysis

In phishing detection:
- False Positive: legitimate URL classified as phishing.
- False Negative: phishing URL classified as legitimate.

False Negatives are especially dangerous because they allow malicious URLs to reach users.

In [ ]:
best_model_name = comparison_df.iloc[0]["model"]

if best_model_name == "Soft Voting Ensemble":
    best_model = voting_model
else:
    best_model = trained_models[best_model_name]

y_pred = best_model.predict(X_test)

error_df = X_test.copy()
error_df["true_label"] = y_test.values
error_df["pred_label"] = y_pred
error_df["url"] = Dataset.loc[X_test.index, "url"].values
error_df["status"] = Dataset.loc[X_test.index, "status"].values

false_positives = error_df[(error_df["true_label"] == 0) & (error_df["pred_label"] == 1)]
false_negatives = error_df[(error_df["true_label"] == 1) & (error_df["pred_label"] == 0)]

print("Best model:", best_model_name)
print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nFalse positives examples:")
display(false_positives[["url", "status", "true_label", "pred_label"]].head(10))

print("\nFalse negatives examples:")
display(false_negatives[["url", "status", "true_label", "pred_label"]].head(10))

# 9. Save Results

In [ ]:
Path("results").mkdir(exist_ok=True)
comparison_df.to_csv("results/model_metrics.csv", index=False)
print("Saved results/model_metrics.csv")

# 10. Initial Conclusions

This adapted experiment follows the methodology of the selected phishing detection project, but runs it on the approved `dataset_phishing.csv` file.

The dataset is balanced and contains engineered URL/webpage features. Multiple models were trained and compared. The final evaluation focuses on precision, recall, F1-score, MCC, ROC-AUC and the confusion matrix, because accuracy alone is not sufficient for phishing detection.

The report should clearly explain that the original notebook could not be executed directly because the original processed CSV was not included in the source repository. Therefore, this notebook adapts the original workflow to the approved CSV dataset.